In [1]:
import os
import glob
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras.saving import register_keras_serializable
from tensorflow.keras import layers, models, Input, callbacks, regularizers, Model
from sklearn.neighbors import KernelDensity
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr
from tensorflow.keras import mixed_precision
from tensorflow.keras import regularizers
mixed_precision.set_global_policy('mixed_float16')
#---------------------------------------------------------------------------------------------------------------------------------
#---configs so I don't have to search and change values at 10 different places in the script everytime I want to change smtg
#---(lesson learnt the hard way)
#----------------------------------------------------------------------------------------------------------------------------------

class Config:
    # --- Paths ---
    H5_PATH = "/kaggle/input/aspcapstar-dr17-150kstars/apogee_dr17_parallel.h5" 
    TFREC_DIR = "/kaggle/working/tfrecords"
    STATS_PATH = "/kaggle/working/dataset_stats.npz"
    IMPUTER_PATH = "/kaggle/working/imputer.pkl"
    IMPUTED_INDICES_PATH = "/kaggle/working/imputed_indices.npz"
    
    # --- System ---
    TRAIN_LIMIT=120_000
    VAL_LIMIT=20_000
    NUM_SHARDS = 16 
    TRAIN_INTEGRATED=True
    TRAIN_BASE_MODEL=True
    RANDOM_SEED=42
    DROP_BAD_DATA = False
    # --- Model Hyperparameters ---
    BATCH_SIZE = 256     
    LEARNING_RATE = 1e-3 
    EPOCHS = 150
    LATENT_DIM = 268
    OUTPUT_LENGTH = 8575
    CALLBACKS=[
            callbacks.ModelCheckpoint('full_model.keras', save_best_only=True, monitor='val_loss'),
            callbacks.EarlyStopping(patience=20, restore_best_weights=True,monitor="val_loss", min_delta=1e-7),
            callbacks.ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-6, verbose=1)
            ]
    # --loss related---
    L2_VAL = 1e-4          
    INPUT_NOISE = 0.05     
    VAR_SCALE = 1e5
    FLOOR_VAR=1e-5
    CLIP_NORM = 1.0     
    BADPIX_CUTOFF=1e-3  
    JACOBIAN_PENALTY_WT=2500
    #----predictor-labels--------
    #CAUTION: LITERALLY EVERYTHING IS IN THE SAME ORDER AS THESE LABELS. DO NOT TOUCH THE ORDER OF THESE LABELS
    SELECTED_LABELS = [
        # 1. Core
        'TEFF', 'LOGG', 'M_H', 'VMICRO', 'VMACRO', 'VSINI',
        # 2. CNO
        'C_FE', 'N_FE', 'O_FE',
        #3. metals
        'FE_H',
        'MG_FE', 'SI_FE', 'CA_FE', 'TI_FE', 'S_FE',
        'AL_FE', 'MN_FE', 'NI_FE', 'CR_FE','K_FE',
        'NA_FE','V_FE',   'CO_FE',
        # engineered (not actually in the dataset)
        
    ]
    #IDX_GLOBAL = [0, 1, 2, 3, 4, 5, 23, 24, 25, 26]  # Teff, Logg, M_H, Vmic, Vmac, Vsini and the other engineered variables
    #IDX_CNO    = [0, 1, 6, 7, 8, 23, 25]
    ABUNDANCE_INDICES =[i for i, label in enumerate(SELECTED_LABELS) if '_FE' in label]
    FE_H_INDEX = SELECTED_LABELS.index('FE_H')
    N_LABELS = len(SELECTED_LABELS) + 4
    #GRAPHING:
    WAVELENGTH_START = 1514
    WAVELENGTH_END = 1694 
    TRAIN_MODEL={'mine':True, "Payne":False, "EXPayne":False, "CNN": False, "RegPayne": False}

config = Config()
np.random.seed(config.RANDOM_SEED)
tf.random.set_seed(config.RANDOM_SEED)
os.makedirs(config.TFREC_DIR, exist_ok=True)

2026-03-31 12:02:09.323687: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774958529.501142      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774958529.553901      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774958530.000171      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774958530.000219      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774958530.000236      24 computation_placer.cc:177] computation placer alr

In [2]:

def _float_feature(value):
    """Returns a float_list from a float / double."""
    return tf.train.Feature(float_list=tf.train.FloatList(value=[value]))

def get_stratified_weights(labels, label_names, bins=15):
    """
    Calculates weights based on 3D density (TEFF, LOGG, M_H).
    """
    try:
        teff = labels[:, label_names.index('TEFF')]
        logg = labels[:, label_names.index('LOGG')]
        m_h = labels[:, label_names.index('M_H')]
    except ValueError:
        print("Warning: TEFF, LOGG, or M_H not found. Returning equal weights.")
        return np.ones(len(labels), dtype=np.float32)

    data = np.vstack([teff, logg, m_h]).T
    hist, edges = np.histogramdd(data, bins=bins)
    weights_map = 1.0 / (hist + 1.0)
    
    idx_x = np.clip(np.digitize(teff, edges[0]) - 1, 0, bins - 1)
    idx_y = np.clip(np.digitize(logg, edges[1]) - 1, 0, bins - 1)
    idx_z = np.clip(np.digitize(m_h,  edges[2]) - 1, 0, bins - 1)

    weights = weights_map[idx_x, idx_y, idx_z]
    clip_threshold = np.percentile(weights, 99)
    weights = np.clip(weights, None, clip_threshold)
    weights = weights / np.mean(weights)

    print(f"  > 3D Stratification: Max W={weights.max():.2f}, Min W={weights.min():.2f}")
    return weights.astype(np.float32)


def engineer_features(labels, selected_labels):
    """
    Applies feature engineering to an already-imputed label array.
    Appends inv_teff, vbroad, C-O, and logPe as new columns.
    Operates purely as a transform — no fitting involved.
    """
    try:
        teff_idx = selected_labels.index('TEFF')
        inv_teff = (5040.0 / labels[:, teff_idx]).reshape(-1, 1)
        labels = np.hstack([labels, inv_teff])
    except ValueError:
        print("TEFF not found in labels")

    try:
        vmacro_idx = selected_labels.index('VMACRO')
        vsini_idx  = selected_labels.index('VSINI')
        vbroad = np.sqrt(
            np.square(labels[:, vmacro_idx]) +
            np.square(labels[:, vsini_idx])
        ).reshape(-1, 1)
        labels = np.hstack([labels, vbroad])
    except ValueError:
        print("VMACRO or VSINI not found in labels")

    try:
        C_idx = selected_labels.index('C_FE')
        O_idx = selected_labels.index('O_FE')
        C_O_diff = (labels[:, C_idx] - labels[:, O_idx]).reshape(-1, 1)
        labels = np.hstack([labels, C_O_diff])
    except ValueError:
        print("C_FE or O_FE not found in labels")

    try:
        G_idx  = selected_labels.index('LOGG')
        MH_idx = selected_labels.index('M_H')
        logPe = (0.5*labels[:, G_idx] + 0.5 * labels[:, MH_idx]).reshape(-1, 1)
        labels = np.hstack([labels, logPe])
    except ValueError:
        print("M_H or LOGG not found in labels")

    return labels


def get_raw_flagged(h5_path, selected_labels, start, end):
    """
    Reads raw label values and constructs bad_mask for a given index slice.
    Extracted so train and val slices can be read independently.
    """
    with h5py.File(h5_path, 'r') as f:
        get_col = lambda k: f['metadata'][k]
        keys = f['metadata'].dtype.names
        print(keys)
        raw_values = np.stack([get_col(p)[start:end] for p in selected_labels], axis=1)
        bad_mask = np.zeros_like(raw_values, dtype=bool)

        for i, label in enumerate(selected_labels):
            flag_name = f"{label}_FLAG"
            if flag_name in keys:
                flg = get_col(flag_name)[start:end]
                if flg.dtype.names: flg = flg[flg.dtype.names[0]]
                if flg.dtype.kind == 'V': flg = flg.view('<i4')
                bad_mask[:, i] = (flg.astype(int) != 0)
            elif label in ['TEFF', 'LOGG', 'VMICRO', 'VMACRO', 'VSINI']:
                bad_mask[:, i] = (raw_values[:, i] < -5000)

    return raw_values, bad_mask


def get_clean_imputed_data(h5_path, selected_labels, train_limit, val_limit, drop_bad_data=False):
    """
    Returns clean, feature-engineered label arrays for train and validation splits.
    
    - Imputer is fit on training data only, then applied to validation.
    - Stats (mean, std) are computed on training data only, then saved.
    - Feature engineering is a pure transform applied identically to both splits.
    
    Args:
        h5_path:        Path to the HDF5 file.
        selected_labels: List of label column names.
        train_limit:    Number of stars to use for training (first N stars).
        val_limit:      Number of stars to use for validation (next M stars).
    
    Returns:
        train_labels, val_labels, train_indices, val_indices
    """

    print(f"Reading training data (stars 0 to {train_limit})...")
    train_raw, train_bad = get_raw_flagged(h5_path, selected_labels, 0, train_limit)

    print(f"Reading validation data (stars {train_limit} to {train_limit + val_limit})...")
    val_raw, val_bad = get_raw_flagged(h5_path, selected_labels, train_limit, train_limit + val_limit)

    def report_and_get_indices(bad_mask, split_name, offset):
        imputed_rows = np.any(bad_mask, axis=1)
        imputed_indices = np.where(imputed_rows)[0]
        
        print(f"--- {split_name} imputed/bad data details ---")
        for local_idx in imputed_indices:
            bad_cols = np.where(bad_mask[local_idx])[0]
            bad_labels = [selected_labels[c] for c in bad_cols]
            print(f"Index {local_idx + offset}: Bad labels (imputed/dropped): {', '.join(bad_labels)}")
            
        return imputed_rows
        
    train_imputed_rows = report_and_get_indices(train_bad, "Train", 0)
    val_imputed_rows = report_and_get_indices(val_bad, "Validation", train_limit)
    
    if drop_bad_data:
        print("Dropping rows with bad data...")
        train_valid = ~train_imputed_rows
        val_valid = ~val_imputed_rows
        
        train_to_impute = train_raw[train_valid].copy().astype(float)
        val_to_impute = val_raw[val_valid].copy().astype(float)
        
        train_indices = np.where(train_valid)[0]
        val_indices = np.where(val_valid)[0] + train_limit
    else:
        train_to_impute = train_raw.copy().astype(float)
        train_to_impute[train_bad] = np.nan
        val_to_impute = val_raw.copy().astype(float)
        val_to_impute[val_bad] = np.nan
        
        train_indices = np.arange(train_limit)
        val_indices = np.arange(train_limit, train_limit + val_limit)

    # --- Imputation: fit on train only ---
    print("Fitting imputer on training data...")

    imputer = IterativeImputer(
        estimator=BayesianRidge(),
        max_iter=10,
        initial_strategy='median'
    )
    train_imputed = imputer.fit_transform(train_to_impute)  # fit + transform
    val_imputed   = imputer.transform(val_to_impute)        # transform only
    
    import joblib
    print(f"Saving imputer to {config.IMPUTER_PATH} ...")
    joblib.dump(imputer, config.IMPUTER_PATH)
    print(f"Saving imputed indices to {config.IMPUTED_INDICES_PATH} ...")
    np.savez(config.IMPUTED_INDICES_PATH, \
        train_bad_mask=train_bad, \
        val_bad_mask=val_bad, \
        train_imputed_values=train_imputed[train_bad] if not drop_bad_data else np.array([]), \
        val_imputed_values=val_imputed[val_bad] if not drop_bad_data else np.array([])\
    )

    # --- [X/Fe] → [X/H] conversion (pure transform, same for both) ---
    print("Converting [X/Fe] to [X/H]...")
    fe_col_train = train_imputed[:, config.FE_H_INDEX]
    fe_col_val   = val_imputed[:,   config.FE_H_INDEX]
    for idx in config.ABUNDANCE_INDICES:
        train_imputed[:, idx] += fe_col_train
        val_imputed[:,   idx] += fe_col_val

    # --- Feature engineering (pure transform, no fitting) ---
    print("Engineering features...")
    train_engineered = engineer_features(train_imputed, selected_labels)
    val_engineered   = engineer_features(val_imputed,   selected_labels)

    # --- Normalisation: compute stats on train only, apply to both ---
    print("Computing normalisation statistics from training data...")
    mean_labels = np.mean(train_engineered, axis=0)
    std_labels  = np.std( train_engineered, axis=0)
    std_labels[std_labels == 0] = 1.0  # avoid divide-by-zero for constant features

    np.savez(config.STATS_PATH, mean=mean_labels, std=std_labels)
    print(f"Stats saved to {config.STATS_PATH}")

    train_labels = (train_engineered - mean_labels) / std_labels
    val_labels   = (val_engineered   - mean_labels) / std_labels

    print(f"Train label shape: {train_labels.shape}")
    print(f"Val   label shape: {val_labels.shape}")

    return train_labels, val_labels, train_indices, val_indices


def _bytes_feature(value):
    if isinstance(value, type(tf.constant(0))): value = value.numpy()
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))


def write_split_to_tfrecords(h5_path, labels, weights, indices, split_name):
    """
    Writes a single split (train or val) to sharded TFRecords.
    Reads the corresponding flux/ivar slice from the HDF5 file.
    """
    total = len(labels)
    shard_size = int(np.ceil(total / config.NUM_SHARDS))

    with h5py.File(h5_path, 'r') as f:
        ds_flux = f['flux']
        ds_ivar = f['ivar']

        for shard_id in range(config.NUM_SHARDS):
            local_start = shard_id * shard_size
            local_end   = min((shard_id + 1) * shard_size, total)
            if local_start >= total:
                break

            global_indices = indices[local_start:local_end]

            filename = os.path.join(
                config.TFREC_DIR,
                f"{split_name}_{shard_id:02d}.tfrec"
            )
            print(f"   Writing {filename} ...")

            idx_list = global_indices.tolist()
            chunk_flux    = ds_flux[idx_list]
            chunk_ivar    = ds_ivar[idx_list]
            chunk_labels  = labels[local_start:local_end]
            chunk_weights = weights[local_start:local_end]

            with tf.io.TFRecordWriter(filename) as writer:
                for i in range(len(chunk_labels)):
                    label_bytes = tf.io.serialize_tensor(
                        chunk_labels[i].astype(np.float32)
                    )
                    spec_raw   = np.stack(
                        [chunk_flux[i], chunk_ivar[i]], axis=-1
                    ).astype(np.float32)
                    spec_bytes = tf.io.serialize_tensor(spec_raw)

                    feature = {
                        'labels':  _bytes_feature(label_bytes),
                        'spectra': _bytes_feature(spec_bytes),
                        'weight':  _float_feature(chunk_weights[i]),
                    }
                    example = tf.train.Example(
                        features=tf.train.Features(feature=feature)
                    )
                    writer.write(example.SerializeToString())


def generate_tfrecords():
    print(f"Starting TFRecord Generation (Shards: {config.NUM_SHARDS})...")

    train_limit = config.TRAIN_LIMIT  # e.g. 120000
    val_limit   = config.VAL_LIMIT    # e.g. 20000
    # Stars beyond train_limit + val_limit are the test set — untouched here

    # --- Labels ---
    train_labels, val_labels, train_indices, val_indices = get_clean_imputed_data(
        config.H5_PATH, config.SELECTED_LABELS, train_limit, val_limit, drop_bad_data=config.DROP_BAD_DATA
    )

    # --- Stratified weights (fit independently per split) ---
    print("Computing stratified weights...")
    train_weights = get_stratified_weights(train_labels, config.SELECTED_LABELS)
    val_weights   = get_stratified_weights(val_labels,   config.SELECTED_LABELS)

    # --- Write TFRecords ---
    print("Writing training TFRecords...")
    write_split_to_tfrecords(
        config.H5_PATH, train_labels, train_weights,
        indices=train_indices, split_name="train"
    )

    print("Writing validation TFRecords...")
    write_split_to_tfrecords(
        config.H5_PATH, val_labels, val_weights,
        indices=val_indices, split_name="val"
    )

    print("TFRecord Generation Complete.")


# Generate on first run
if not glob.glob(os.path.join(config.TFREC_DIR, "*.tfrec")):
    generate_tfrecords()
else:
    print("TFRecords found. Skipping generation.")

Starting TFRecord Generation (Shards: 16)...
Reading training data (stars 0 to 120000)...
('FILE', 'APOGEE_ID', 'TARGET_ID', 'APSTAR_ID', 'ASPCAP_ID', 'TELESCOPE', 'LOCATION_ID', 'FIELD', 'ALT_ID', 'RA', 'DEC', 'GLON', 'GLAT', 'J', 'J_ERR', 'H', 'H_ERR', 'K', 'K_ERR', 'SRC_H', 'WASH_M', 'WASH_M_ERR', 'WASH_T2', 'WASH_T2_ERR', 'DDO51', 'DDO51_ERR', 'IRAC_3_6', 'IRAC_3_6_ERR', 'IRAC_4_5', 'IRAC_4_5_ERR', 'IRAC_5_8', 'IRAC_5_8_ERR', 'IRAC_8_0', 'IRAC_8_0_ERR', 'WISE_4_5', 'WISE_4_5_ERR', 'TARG_4_5', 'TARG_4_5_ERR', 'WASH_DDO51_GIANT_FLAG', 'WASH_DDO51_STAR_FLAG', 'TARG_PMRA', 'TARG_PMDEC', 'TARG_PM_SRC', 'AK_TARG', 'AK_TARG_METHOD', 'AK_WISE', 'SFD_EBV', 'APOGEE_TARGET1', 'APOGEE_TARGET2', 'APOGEE2_TARGET1', 'APOGEE2_TARGET2', 'APOGEE2_TARGET3', 'APOGEE2_TARGET4', 'TARGFLAGS', 'SURVEY', 'PROGRAMNAME', 'NVISITS', 'SNR', 'SNREV', 'STARFLAG', 'STARFLAGS', 'ANDFLAG', 'ANDFLAGS', 'VHELIO_AVG', 'VSCATTER', 'VERR', 'RV_TEFF', 'RV_LOGG', 'RV_FEH', 'RV_ALPHA', 'RV_CARB', 'RV_CHI2', 'RV_CCFWHM', 'R

I0000 00:00:1774958613.098556      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


   Writing /kaggle/working/tfrecords/train_01.tfrec ...
   Writing /kaggle/working/tfrecords/train_02.tfrec ...
   Writing /kaggle/working/tfrecords/train_03.tfrec ...
   Writing /kaggle/working/tfrecords/train_04.tfrec ...
   Writing /kaggle/working/tfrecords/train_05.tfrec ...
   Writing /kaggle/working/tfrecords/train_06.tfrec ...
   Writing /kaggle/working/tfrecords/train_07.tfrec ...
   Writing /kaggle/working/tfrecords/train_08.tfrec ...
   Writing /kaggle/working/tfrecords/train_09.tfrec ...
   Writing /kaggle/working/tfrecords/train_10.tfrec ...
   Writing /kaggle/working/tfrecords/train_11.tfrec ...
   Writing /kaggle/working/tfrecords/train_12.tfrec ...
   Writing /kaggle/working/tfrecords/train_13.tfrec ...
   Writing /kaggle/working/tfrecords/train_14.tfrec ...
   Writing /kaggle/working/tfrecords/train_15.tfrec ...
Writing validation TFRecords...
   Writing /kaggle/working/tfrecords/val_00.tfrec ...
   Writing /kaggle/working/tfrecords/val_01.tfrec ...
   Writing /kaggle/w

In [3]:
# Load Stats
stats = np.load(config.STATS_PATH)
MEAN_TENSOR = tf.constant(stats['mean'], dtype=tf.float32)
STD_TENSOR = tf.constant(stats['std'], dtype=tf.float32)

# ==========================================
#          NORMALISE THE DATA
# ==========================================
def parse_and_normalize(example_proto):
    feature_desc = {
        'labels': tf.io.FixedLenFeature([], tf.string),
        'spectra': tf.io.FixedLenFeature([], tf.string),
        'weight': tf.io.FixedLenFeature([], tf.float32) # <--- Read Weight
    }
    
    parsed = tf.io.parse_single_example(example_proto, feature_desc)
    
    # Parse Tensors
    labels = tf.io.parse_tensor(parsed['labels'], out_type=tf.float32)
    spectra = tf.io.parse_tensor(parsed['spectra'], out_type=tf.float32)
    weight = parsed['weight'] #already parsed
    
    labels.set_shape([config.N_LABELS])
    spectra.set_shape([config.OUTPUT_LENGTH, 2])
    
    # Normalize Labels
    #norm_labels = (labels - MEAN_TENSOR) / STD_TENSOR
    
    # Return (Input, Target, Sample_Weight)
    return labels, spectra, weight
def build_dataset():
    train_files = sorted(tf.io.gfile.glob(os.path.join(config.TFREC_DIR, "train_*.tfrec")))
    val_files   = sorted(tf.io.gfile.glob(os.path.join(config.TFREC_DIR, "val_*.tfrec")))
    print(f"Data Split: {len(train_files)} Train Files, {len(val_files)} Val Files")
    
    def load_files(filenames):
        ds = tf.data.TFRecordDataset(filenames, num_parallel_reads=tf.data.AUTOTUNE)
        ds = ds.map(parse_and_normalize, num_parallel_calls=tf.data.AUTOTUNE)
        return ds
    
    train_ds = load_files(train_files).shuffle(10000).batch(config.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    val_ds = load_files(val_files).batch(config.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return train_ds, val_ds

train_ds, val_ds = build_dataset()

Data Split: 16 Train Files, 16 Val Files


In [4]:
import numpy as np

def generate_jacobian_mask(selected_labels, n_pixels=config.OUTPUT_LENGTH, start_lambda=config.WAVELENGTH_START, end_lambda=config.WAVELENGTH_END):
  
    wavelengths = np.logspace(np.log10(start_lambda),np.log10(end_lambda),n_pixels)
    #wavelengths in vacuum nm
    # Source: Shetrone et al. 2015 & Smith et al. 2021 (APOGEE DR17)
  
    
    features = {
        # --- Group 1: core ---
        'TEFF': [], 'LOGG': [], 'M_H': [],  
        'VMICRO': [], 'VMACRO': [], 'VSINI': [],
        'INV_TEFF':[], 'vbroad':[], 'LOGPE':[],

        # --- Group 2: CNO ---
        # Kept global because they form the pseudo-continuum haze.
        'C_FE':  [], 
        'N_FE':  [],                  
        'O_FE':  [], 
        'C_O_diff':[], # Helper for C/O ratio physics
        
        # --- Group 3: other Abundances ---
        'FE_H': [],
           # 1519.45, 1520.75, 1529.46, 1538.20, # Blue side anchors
           # 1539.57, 1549.03, 1559.15, 
           # 1569.28, # The famous "Magnetic" line (very strong)
           # 1583.52, 1596.49, 1600.96, 
          #  1615.32, 1616.50, 
         #   1627.32, 1636.23, 1642.46,          
        #    1655.22, 1661.26, 1680.20           
        #],

        'MG_FE': [],#[1574.07, 1574.88, 1576.58, 1595.88], # the triplet, and the other line
        'SI_FE': [],#[1596.01, 1606.00, 1609.48, 1621.57, 1668.08],
        'CA_FE': [],#[1615.08, 1615.74, 1619.70],
        'TI_FE': [],#[1560.28, 1569.89, 1571.55, 1663.50],
        'S_FE':  [],#[1540.30, 1542.20, 1546.98, 1547.85],
        'AL_FE': [],#[1671.89, 1675.06, 1676.34],          
        'MN_FE': [],#[1515.90, 1521.70, 1526.20],
        'NI_FE': [],#[1563.20, 1658.40, 1667.30, 1681.50],
        'CR_FE': [],#[1568.00],                    
        'K_FE':  [],#[1516.31, 1516.84],                   
        'NA_FE': [],#[1637.38, 1638.88],                   
        'V_FE':  [],#[1592.40],
        'CO_FE': [],#[1675.77]#meh
    }

    #  Initialize Matrix (Default to 1.0 = Forbidden)
    mask_matrix = np.ones((n_pixels, len(selected_labels)), dtype=np.float32)
  
    window_width = 0.2 #0.2 nm is big enough to catch small changes 
    
    print(f"{'Label':<10} | {'Status':<10} | {'Allowed Pixels'}")
    print("-" * 40)

    for i, label in enumerate(selected_labels):

        # no mask for global
        if label in features:
            mask_matrix[:, i] = 0.0
            print(f"{label:<10} | GLOBAL     | 8575 (All)")
            continue
            
        # Scenario B: Specific Elements
        if label in features:
            lines = features[label]
                
            for center in lines:
                # Find indices within +/- window of the line center (0.2nm)
                indices = np.where(np.abs(wavelengths - center) < window_width)[0]
                mask_matrix[indices, i] = 0.0 # Set to Allowed
            
            # Count allowed pixels
            allowed_count = np.sum(mask_matrix[:, i] == 0.0)
            print(f"{label:<10} | MASKED     | {int(allowed_count)}")
            
        else:
            print(f"{label:<10} | you messed up here pal  | 0 (All Forbidden)")

    return mask_matrix

# --- USE IT ---
SELECTED_LABELS = config.SELECTED_LABELS+['INV_TEFF','vbroad','C_O_diff','LOGPE']


final_mask = generate_jacobian_mask(SELECTED_LABELS)
np.save("jacobian_mask.npy", final_mask)
print("\nMask saved to 'jacobian_mask.npy'")

Label      | Status     | Allowed Pixels
----------------------------------------
TEFF       | GLOBAL     | 8575 (All)
LOGG       | GLOBAL     | 8575 (All)
M_H        | GLOBAL     | 8575 (All)
VMICRO     | GLOBAL     | 8575 (All)
VMACRO     | GLOBAL     | 8575 (All)
VSINI      | GLOBAL     | 8575 (All)
C_FE       | GLOBAL     | 8575 (All)
N_FE       | GLOBAL     | 8575 (All)
O_FE       | GLOBAL     | 8575 (All)
FE_H       | GLOBAL     | 8575 (All)
MG_FE      | GLOBAL     | 8575 (All)
SI_FE      | GLOBAL     | 8575 (All)
CA_FE      | GLOBAL     | 8575 (All)
TI_FE      | GLOBAL     | 8575 (All)
S_FE       | GLOBAL     | 8575 (All)
AL_FE      | GLOBAL     | 8575 (All)
MN_FE      | GLOBAL     | 8575 (All)
NI_FE      | GLOBAL     | 8575 (All)
CR_FE      | GLOBAL     | 8575 (All)
K_FE       | GLOBAL     | 8575 (All)
NA_FE      | GLOBAL     | 8575 (All)
V_FE       | GLOBAL     | 8575 (All)
CO_FE      | GLOBAL     | 8575 (All)
INV_TEFF   | GLOBAL     | 8575 (All)
vbroad     | GLOBAL     | 8575

In [5]:
@register_keras_serializable()
def spl_loss(y_true, y_pred):
    if len(y_pred.shape) == 2:
        y_pred = tf.expand_dims(y_pred, -1)
    real_flux = y_true[:, :, 0:1]
    ivar = y_true[:, :, 1:2]
    valid_mask = tf.cast(real_flux > config.BADPIX_CUTOFF, tf.float32)    
    safe_flux = tf.where(valid_mask == 1.0, real_flux, y_pred)
    #ivar_safe = ivar/(config.IVAR_SCALE+ivar)# scale and soft 
    weight=1e-5/(1/ivar+1e-5)
    weight=tf.where(ivar>0,weight,0.0)#weight=tf.where(((safe_flux<0.9) & (ivar>0)),tf.maximum(ivar_safe,tf.cast(1.0,dtype=tf.float32)),ivar_safe)
    #chi2
    wmse_term = tf.square(safe_flux - y_pred) * weight * valid_mask
    
    loss = tf.where(tf.math.is_finite(wmse_term), wmse_term, tf.zeros_like(wmse_term))
    
    return loss

@register_keras_serializable()
def chi2_estimate(y_true,y_pred):
    y_true=tf.cast(y_true,tf.float32)
    y_pred=tf.cast(y_pred,tf.float32)
    if len(y_pred.shape) == 2:
        y_pred = tf.expand_dims(y_pred, -1)
    real_flux = y_true[:, :, 0:1]
    ivar = y_true[:, :, 1:2]
    valid_mask = tf.cast(real_flux > config.BADPIX_CUTOFF, tf.float32)    
    safe_flux = tf.where(valid_mask == 1.0, real_flux, y_pred)
    #ivar_safe = ivar/(config.IVAR_SCALE+ivar)# scale and soft 
    weight=ivar
    weight=tf.where(ivar>0,weight,0.0)#weight=tf.where(((safe_flux<0.9) & (ivar>0)),tf.maximum(ivar_safe,tf.cast(1.0,dtype=tf.float32)),ivar_safe)
    #chi2
    wmse_term = tf.square(safe_flux - y_pred) * weight * valid_mask
    n_valid_pixels = tf.reduce_sum(valid_mask, 1)
    n_params = len(config.SELECTED_LABELS)
    dof = n_valid_pixels - n_params
    return tf.reduce_sum(wmse_term,1)/dof

In [6]:


class StochasticPerturbation(layers.Layer):
    def __init__(self, mask_matrix, epsilon=0.05, **kwargs):
        super().__init__(**kwargs)
        # mask_matrix shape: (8575, 22) -> We transpose to (22, 8575) for easier gathering
        self.mask_matrix = tf.constant(mask_matrix.T, dtype=tf.float32) 
        self.epsilon = epsilon
        self.original_mask = tf.constant(mask_matrix, dtype=tf.float32)
        
        #Pick only indeces which have some mask
        self.constrained_indices = tf.where(tf.reduce_sum(self.original_mask, axis=0) > 0)[:, 0]
        self.num_constrained = tf.shape(self.constrained_indices)[0]

    def call(self, inputs, training=None):
        #Only run this if you are training
        if not training:
            #return dummy data for graph connection
            return inputs, tf.zeros((tf.shape(inputs)[0], tf.shape(self.original_mask)[0]))

        batch_size = tf.shape(inputs)[0]
        
        # 1. Roulette: Pick one random index per star
        random_choices = tf.random.uniform(
            shape=(batch_size,), 
            minval=0, 
            maxval=self.num_constrained, 
            dtype=tf.int32
        )
        # Map to real label indices
        target_labels = tf.gather(self.constrained_indices, random_choices)
        
        # 2. Create Wiggle Vectors
        wiggle_vectors = tf.one_hot(
            target_labels, 
            depth=tf.shape(inputs)[1], 
            on_value=self.epsilon, 
            off_value=0.0
        )
        perturbed_inputs = inputs + wiggle_vectors
        
        # 3. Get the Corresponding Masks
        # mask matrix: (8575, 22) -> Transpose to (22, 8575)
        mask_T = tf.transpose(self.original_mask) 
        # Gather rows
        active_masks = tf.gather(mask_T, target_labels)
        
        return perturbed_inputs, active_masks

class PhysicsLossLayer(layers.Layer):

    def __init__(self, penalty_weight=config.JACOBIAN_PENALTY_WT, **kwargs):
        super().__init__(**kwargs)
        self.penalty_weight = penalty_weight

    def call(self, inputs):
        # Unpack inputs: [Clean_Spectrum, Perturbed_Spectrum, Active_Masks]
        y_clean, y_perturbed, active_masks = inputs[0], inputs[1], inputs[2]
        
        # Calculate Difference
        diff = tf.abs(y_clean - y_perturbed)
        
        # Apply Mask (Diff * 1.0 if forbidden, 0.0 if allowed)
        # active_masks shape is (Batch, Pixels)
        penalty_map = diff * active_masks
        
        # Calculate Scalar Loss
        physics_loss = tf.reduce_mean(penalty_map) * self.penalty_weight
        
        # MAGIC: Inject this loss into the main training loop
        self.add_loss(physics_loss)
        
        
        return y_clean # Pass through clean output

In [7]:
def build_disentangled_model(base_generator, mask_matrix, penalty_weight=100):
    
    # 1. Input Layer
    inputs = Input(shape=(config.N_LABELS,), name="labels")
    
    # 2. Main Pass (The Normal Path)
    # We call the generator once to get the "Real" prediction
    clean_output = base_generator(inputs)
    
    # 3. The Shadow Pass (The Physics Check)
    # A. Generate wiggled inputs + relevant masks
    perturbed_inputs, active_masks = StochasticPerturbation(mask_matrix)(inputs)
    
    # B. Run the generator AGAIN on wiggled inputs
    # Note: 'base_generator' weights are SHARED automatically!
    perturbed_output = base_generator(perturbed_inputs)
    
    # C. Calculate and Add Loss
    # We pass all 3 tensors into the loss layer
    # This layer returns 'clean_output' just to complete the graph flow, 
    # but primarily it adds the loss via side-effect.
    _ = PhysicsLossLayer(penalty_weight)([clean_output, perturbed_output, active_masks])
    
    # 4. Construct Final Model
    # We only output 'clean_output'. The user never sees the shadow branch.
    model = Model(inputs=inputs, outputs=clean_output)
    
    return model

In [8]:


# ==========================================
# 1. CONFIGURATION & INDICES
# ==========================================


# Indices for slicing inputs (Hard-coded based on LABELS_ORDER)
IDX_GLOBAL = [0, 1, 2, 3, 4, 5, 23, 24, 25, 26]  # Teff, Logg, M_H, Vmic, Vmac, Vsini and the other engineered variables
IDX_CNO    = [0, 1, 6, 7, 8, 23, 25] # Global + C, N, O, CO (Molecules need Teff/Logg/Fe too)

# Elements to treat as Independent Sparse Experts

LABELS_ORDER = config.SELECTED_LABELS+ ['INV_TEFF', 'vbroad', 'C_O_diff', 'LOGPE']             #22-25: engineered 
# Elements to treat as Independent Sparse Experts
ATOMIC_LABELS = [
   'FE_H',
   'MG_FE', 'SI_FE', 'CA_FE', 'TI_FE', 'S_FE',
   'AL_FE', 'MN_FE', 'NI_FE', 'CR_FE','K_FE',
   'NA_FE','V_FE','CO_FE',
]

In [ ]:
@tf.keras.utils.register_keras_serializable()
class ColumnSelector(layers.Layer):
    """
    layer that selects specific columns from the input.
    """
    def __init__(self, indices, **kwargs):
        super().__init__(**kwargs)
        self.indices = list(indices) 

    def call(self, inputs):
        return tf.gather(inputs, self.indices, axis=1)

    def get_config(self):
        config = super().get_config()
        config.update({'indices': self.indices})
        return config

@tf.keras.utils.register_keras_serializable()
class BeerLambertLaw(layers.Layer):
    """
    Applies the Beer-Lambert law: Flux = exp(-Tau).
    adds 1.2 scaling factor
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self,k, tau):
        return k*tf.math.exp(-tau)

@tf.keras.utils.register_keras_serializable()
class GetAbundances(layers.Layer):
    """
    Returns a specifc column ID
    """
    def __init__(self,col_id, **kwargs):
        super().__init__(**kwargs)
        self.index=col_id

    def call(self, inputs):
        return inputs[:, self.index:self.index+1]

@tf.keras.utils.register_keras_serializable()
class SparseProjector(tf.keras.layers.Layer):
    def __init__(self, active_indices, weights, total_pixels, label_name="unknown", **kwargs):
        # Override name before passing to super to avoid conflict in from_config
        kwargs['name'] = f"Sparse_Projector_{label_name}"
        super().__init__(**kwargs)
        self.total_pixels = total_pixels
        self.label_name = label_name
        self.active_indices = tf.constant(active_indices, dtype=tf.int32)
        self.weights_tensor = tf.constant(weights, dtype=tf.float32)

    def call(self, local_tau):
        input_dtype = local_tau.dtype
        local_tau_32 = tf.cast(local_tau, tf.float32)
        weighted = local_tau_32 * self.weights_tensor[None, :]

        batch_size = tf.shape(local_tau)[0]
        n_active = tf.shape(self.active_indices)[0]

        batch_ids = tf.repeat(tf.range(batch_size), n_active)
        pixel_ids = tf.tile(self.active_indices, tf.expand_dims(batch_size, axis=0))

        scatter_idx = tf.stack([batch_ids, pixel_ids], axis=-1)
        flat_weighted = tf.reshape(weighted, [-1])

        full = tf.scatter_nd(
            scatter_idx,
            flat_weighted,
            shape=tf.stack([batch_size, tf.constant(self.total_pixels, dtype=tf.int32)])
        )
        # Cast back to original input dtype
        return tf.cast(full, input_dtype)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.total_pixels)

    def get_config(self):
        config = super().get_config()
        config.update({
            'active_indices': self.active_indices.numpy().tolist(),
            'weights': self.weights_tensor.numpy().tolist(),
            'total_pixels': self.total_pixels,
            'label_name': self.label_name,
        })
        return config

    @classmethod
    def from_config(cls, config):
        config['active_indices'] = np.array(config['active_indices'], dtype=np.int32)
        config['weights'] = np.array(config['weights'], dtype=np.float32)
        config.pop('name', None)  # remove saved name, __init__ constructs it from label_name
        return cls(**config)

In [10]:
# ==========================================
# 2. MASK LOADER
# ==========================================
def load_element_indices(mask_path='jacobian_mask.npy', threshold=0.01):
  
    print(f"Loading mask from {mask_path}...")
    mask = np.load(mask_path)
    # jacobian_mask uses inverted convention (0=allowed, 1=forbidden)
    # Convert to weight convention (0=forbidden, higher=allowed)
    if 'jacobian' in mask_path:
        mask = 1.0 - mask
    indices_dict = {}
    
    for label in ATOMIC_LABELS:
        if label not in LABELS_ORDER: continue
        col_idx = LABELS_ORDER.index(label)
        
        # Logic: higher values = more influence, 0.0 = forbidden 
        active_pixels = np.where(mask[:, col_idx] > threshold)[0]
        
        if len(active_pixels) > 0:
            indices_dict[label] = (active_pixels, mask[active_pixels, col_idx])
            print(f"  > {label}: Found {len(active_pixels)} active pixels.")
            
    return indices_dict

# =========================================================
#  layer that masks out outputs in every non-masked pixel
# =========================================================

def create_sparse_projector(active_indices, weights, total_pixels=config.OUTPUT_LENGTH, label_name="unknown"):
    return SparseProjector(
        active_indices=active_indices.astype(np.int32),   # explicit int32
        weights=weights.astype(np.float32),               # explicit float32
        total_pixels=total_pixels,
        label_name=label_name
    )
    

def build_continuum_branch(input_global):
    # Start with a Low res base: (268)
    x = layers.Dense(268, activation='swish')(input_global)
    x = layers.Reshape((268, 1))(x)
    
    # 2. Progressive Upsampling Loop (32x total)
    # We use 5 blocks: 268 -> 536 -> 1072 -> 2144 -> 4288 -> 8576
    filters = [16, 16, 32, 32, 32] # Gradually increase capacity or keep constant
    
    for f in filters:
        x = layers.UpSampling1D(size=2)(x)
        x = layers.Conv1D(f, kernel_size=3, padding='same')(x)
        x = layers.LayerNormalization()(x) # Stabilizes the deep network
        x = layers.Activation('swish')(x)
    
    # 3. Final Smoothing (Physics Layer)
    # Collapses filters to 1 channel (Opacity)
    # Kernel 51 prevents high-frequency noise (mimicking lines)
    x = layers.Conv1D(1, 51, padding='same', activation='softplus', name="K_Continuum_PreCrop")(x)
    
    # 4. Crop to exact size
    # 8576 -> 8575
    k_cont = layers.Cropping1D(cropping=(0, 1), name="K_Continuum")(x)
    
    return layers.Flatten()(k_cont)
    
# ==========================================
#    FINAL MODEL (With Internal Splitting)
# ==========================================  
def build_final_emulator(element_indices):
    
    # --- SINGLE INPUT ENTRY POINT ---
    full_input = Input(shape=(config.N_LABELS,), name="All_Labels")

    # --- 1. INTERNAL SLICING & CONVERSION ---
    # Slice A: Global Params (Teff, Logg, M_H...)
    input_global = ColumnSelector(IDX_GLOBAL, name="Slice_Global")(full_input)
    
    # Slice B: CNO Params
    
    input_cno = ColumnSelector(IDX_CNO, name="Slice_CNO")(full_input)
    k_cont = build_continuum_branch(input_global)

    # --- 3. BRANCH B: MOLECULES ---
    x_mol = layers.Dense(256, activation='swish')(input_cno)
    x_mol = layers.Dense(1024, activation='swish')(x_mol)
    tau_mol = layers.Dense(8575, activation='softplus', name="Tau_Molecules")(x_mol)

    #---- BRANCH C: Thermodynamic state------
    input_EOS = ColumnSelector(IDX_GLOBAL + IDX_CNO, name="Slice_Global_Full")(full_input)

    # 2. THE THERMODYNAMIC STATE BRANCH (EOS)
    # Now it sees the FULL physics context (Atmosphere + CNO + Broadening)
    x_state = layers.Dense(32, activation='swish', name="EOS_Hidden_1")(input_EOS)
    x_state = layers.Dense(16, activation='swish', name="EOS_Hidden_2")(x_state)
    
    # Latent Vector: Represents "Optical Depth Scale" & "Ionization State"
    state_vector = layers.Dense(8, activation='linear', name="State_Vector")(x_state)

    # --- BRANCH D: SPARSE ATOMIC EXPERTS ---
    tau_atoms = []
    
    for label, (indices, weights) in element_indices.items():
        n_active = len(indices)
        col_idx = LABELS_ORDER.index(label)
        
        # A. Slice the specific element [X/Fe]
        raw_abundance =GetAbundances(col_idx,name=f"Slice_{label}")(full_input)

        expert_in = layers.Concatenate()([state_vector, raw_abundance])
        
        # D. Expert Brain
        x = layers.Dense(64, activation='swish')(expert_in)
        local_tau = layers.Dense(n_active, activation='softplus', name=f"Tau_{label}_Local")(x)
        
        # E. Sparse Projector (weighted)
        projector = create_sparse_projector(indices, weights, label_name=label)
        full_tau = projector(local_tau)
        tau_atoms.append(full_tau)

    # --- 5. BRANCH D: RESIDUAL SOLVER ---
    '''x_trash = layers.Dense(128, activation='swish')(full_input)
    tau_residual = layers.Dense(config.OUTPUT_LENGTH, activation='softplus', 
                                dtype='float32',
                                activity_regularizer=regularizers.l1(1e-4),
                                name="Tau_Residual")(x_trash)'''
    # --- 6. PHYSICS SUMMATION ---
    all_sources = [tau_mol,] + tau_atoms
    total_tau = layers.Add(name="Total_Opacity")(all_sources)
    
    # --- FIX IS HERE ---
    # Wrap the math in a Lambda layer to satisfy Keras
    output_flux = BeerLambertLaw(dtype='float32', name="Final_Flux")(k_cont,total_tau)

    return models.Model(inputs=full_input, outputs=output_flux, name="Sparse_Physics_Emulator")

In [ ]:

# Ensure datasets are loaded before fine-tuning
train_ds, val_ds = build_datasets(config.H5_PATH)
mask = np.load("/kaggle/input/datasets/aneeshshastri/element-masks/apogee_mask.npy")



In [ ]:
# ============================================================================
# OPTUNA FINE-TUNING STUDIES
# ============================================================================
import optuna
from optuna.integration import TFKerasPruningCallback
import tensorflow as tf
from tensorflow.keras import layers, Model

print("Installing Optuna if running in standard Kaggle env... (!pip install optuna if needed)")

# ----------------------------------------------------------------------------
# 1. EXPANDED PAYNE TUNING
# ----------------------------------------------------------------------------
def build_tunable_payne(n_labels, n_pixels, depth, width):
    inputs = layers.Input(shape=(n_labels,), name="labels")
    x = inputs
    for i in range(depth):
        x = layers.Dense(width, activation="relu", name=f"dense_{i+1}")(x)
    outputs = layers.Dense(n_pixels, activation=None, name="output_flux")(x)
    return Model(inputs=inputs, outputs=outputs, name="TunablePayne")

def objective_expanded_payne(trial):
    depth = trial.suggest_int('depth', 1, 5)
    width = trial.suggest_int('width', 200, 1500)
    
    model = build_tunable_payne(config.N_LABELS, config.OUTPUT_LENGTH, depth=depth, width=width)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(config.LEARNING_RATE, clipnorm=config.CLIP_NORM),
        loss=spl_loss
    )
    
    # We use a smaller epochs boundary but still full 35 to allow pruning
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        TFKerasPruningCallback(trial, 'val_loss')
    ]
    
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=35,
        verbose=0,
        callbacks=callbacks
    )
    
    # Return best val_loss
    return min(history.history['val_loss'])

# ----------------------------------------------------------------------------
# 2. REG PAYNE TUNING
# ----------------------------------------------------------------------------
def objective_reg_payne(trial):
    penalty_1 = trial.suggest_float('penalty_1', 1e-2, 1e2, log=True)
    penalty_2 = trial.suggest_float('penalty_2', 50.0, 2000.0, log=True)
    
    # Base structure is fixed payload for RegPayne, or we can use the best configuration from Expanded Payne.
    # Here we stick to original RegPayne base dimensionality (or Payne2019 base dimensionality).
    # Payne default: depth=2, width=300
    base_model = build_tunable_payne(config.N_LABELS, config.OUTPUT_LENGTH, depth=2, width=300)
    
    # Phase 1: Learn spectral lines loosely
    trainer_phase_1 = build_disentangled_model(base_model, mask_matrix=mask, penalty_weight=penalty_1)
    trainer_phase_1.compile(
        optimizer=tf.keras.optimizers.Adam(config.LEARNING_RATE, clipnorm=config.CLIP_NORM), 
        loss=spl_loss
    )
    
    trainer_phase_1.fit(
        train_ds,
        validation_data=val_ds,
        epochs=5,
        verbose=0
    )
    
    # Phase 2: Discipline via heavy penalty
    # Re-wrap the SAME base_model
    trainer_phase_2 = build_disentangled_model(base_model, mask_matrix=mask, penalty_weight=penalty_2)
    trainer_phase_2.compile(
        optimizer=tf.keras.optimizers.Adam(config.LEARNING_RATE, clipnorm=config.CLIP_NORM), 
        loss=spl_loss
    )
    
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        TFKerasPruningCallback(trial, 'val_loss')
    ]
    
    history = trainer_phase_2.fit(
        train_ds,
        validation_data=val_ds,
        epochs=30,
        verbose=0,
        callbacks=callbacks
    )
    
    return min(history.history['val_loss'])

# ============================================================================
# EXECUTION
# ============================================================================
if __name__ == "__main__":
    print("--- Starting Expanded Payne Tuning ---")
    study_ep = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner(n_warmup_steps=5))
    study_ep.optimize(objective_expanded_payne, n_trials=30)
    print("Best Expanded Payne Params:", study_ep.best_params)
    
    print("\n--- Starting Reg Payne Tuning ---")
    study_rp = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner(n_warmup_steps=5))
    # We prune phase 2
    study_rp.optimize(objective_reg_payne, n_trials=30)
    print("Best Reg Payne Params:", study_rp.best_params)
    
    # Save the studies
    import joblib
    joblib.dump(study_ep, "expanded_payne_study.pkl")
    joblib.dump(study_rp, "reg_payne_study.pkl")



In [ ]:
# ============================================================================
# OPTUNA VISUALIZATIONS
# ============================================================================
# Uncomment these to view plots in Kaggle
# import optuna.visualization as vis
# vis.plot_optimization_history(study_ep).show()
# vis.plot_param_importances(study_ep).show()
# vis.plot_contour(study_ep).show()

# vis.plot_optimization_history(study_rp).show()
# vis.plot_param_importances(study_rp).show()
# vis.plot_contour(study_rp).show()

